# Training Diagnostics - Troubleshoot Poor Performance

**Diagnose why model performance is poor**

Current metrics:
- PR-AUC: 0.5585
- AUROC: 0.6569 (barely better than random 0.5)
- Brier: 0.2535

## Potential Issues to Check:
1. Dataset size and quality
2. Label correctness and distribution
3. Image preprocessing and normalization
4. Class imbalance
5. Model actually learning (training loss decreasing?)
6. Data leakage or incorrect aggregation

---

## Configuration

In [ ]:
BASE_DIR = '/content/drive/MyDrive/vindr-mammo'
PREPROCESSED_DIR = f'{BASE_DIR}/preprocessed_png_512'
TRAIN_CSV = f'{PREPROCESSED_DIR}/train.csv'
VAL_CSV = f'{PREPROCESSED_DIR}/val.csv'
TEST_CSV = f'{PREPROCESSED_DIR}/test.csv'

print("✅ Configuration loaded")

## Step 1: Check Dataset Size and Distribution

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load metadata
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("="*70)
print("DATASET SIZE ANALYSIS")
print("="*70)

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{name} Set:")
    print(f"  Total images: {len(df)}")
    print(f"  Malignant: {(df['label'] == 1).sum()} ({(df['label'] == 1).sum()/len(df)*100:.1f}%)")
    print(f"  Benign:    {(df['label'] == 0).sum()} ({(df['label'] == 0).sum()/len(df)*100:.1f}%)")
    
    if 'study_id' in df.columns:
        print(f"  Patients:  {df['study_id'].nunique()}")
    if 'breast_birads' in df.columns:
        print(f"\n  BI-RADS distribution:")
        print(df['breast_birads'].value_counts())

print("\n" + "="*70)

# ⚠️ WARNING CHECK
if len(train_df) < 500:
    print("\n⚠️  WARNING: Training set is VERY SMALL (<500 images)")
    print("   This is likely the main cause of poor performance!")
    print("   Recommendation: Use more data or reduce model complexity")

if (train_df['label'] == 1).sum() < 50:
    print("\n⚠️  WARNING: Very few malignant samples (<50)")
    print("   Model will struggle to learn cancer patterns!")
    print("   Recommendation: Increase malignant samples or use class weights")

# Check class imbalance
imbalance_ratio = (train_df['label'] == 0).sum() / (train_df['label'] == 1).sum()
print(f"\nClass Imbalance Ratio (Benign:Malignant): {imbalance_ratio:.2f}:1")
if imbalance_ratio > 5:
    print("⚠️  WARNING: Severe class imbalance!")
    print("   Recommendation: Use weighted loss or focal loss")

## Step 2: Verify Images Exist and Are Valid

In [ ]:
import os
from PIL import Image

print("Checking if images exist and are loadable...\n")

def check_images(df, name, max_check=100):
    missing = 0
    corrupted = 0
    valid = 0
    
    check_count = min(len(df), max_check)
    
    for idx in range(check_count):
        row = df.iloc[idx]
        
        # Get image path
        if 'png_path' in row:
            img_path = os.path.join(PREPROCESSED_DIR, row['png_path'])
        elif 'image_path' in row:
            img_path = os.path.join(PREPROCESSED_DIR, row['image_path'])
        else:
            print(f"⚠️  No image path column found!")
            return
        
        # Check existence
        if not os.path.exists(img_path):
            missing += 1
            if missing <= 3:
                print(f"❌ Missing: {img_path}")
            continue
        
        # Try loading
        try:
            img = Image.open(img_path)
            img.verify()  # Verify it's valid
            valid += 1
        except Exception as e:
            corrupted += 1
            if corrupted <= 3:
                print(f"❌ Corrupted: {img_path} - {e}")
    
    print(f"\n{name} Set (checked {check_count} images):")
    print(f"  Valid:     {valid} ({valid/check_count*100:.1f}%)")
    print(f"  Missing:   {missing} ({missing/check_count*100:.1f}%)")
    print(f"  Corrupted: {corrupted} ({corrupted/check_count*100:.1f}%)")
    
    if missing > 0 or corrupted > 0:
        print(f"\n⚠️  WARNING: Found {missing + corrupted} problematic images!")
        return False
    return True

train_ok = check_images(train_df, "Train", max_check=100)
val_ok = check_images(val_df, "Val", max_check=50)

if not train_ok or not val_ok:
    print("\n⚠️  ACTION NEEDED: Fix missing/corrupted images before training!")

## Step 3: Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt

# Sample images from train set
malignant_samples = train_df[train_df['label'] == 1].head(4)
benign_samples = train_df[train_df['label'] == 0].head(4)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Training Images (Top: Malignant, Bottom: Benign)', fontsize=14, fontweight='bold')

def load_and_show(ax, row, title):
    try:
        if 'png_path' in row:
            img_path = os.path.join(PREPROCESSED_DIR, row['png_path'])
        else:
            img_path = os.path.join(PREPROCESSED_DIR, row['image_path'])
        
        img = Image.open(img_path)
        img_array = np.array(img)
        
        ax.imshow(img_array, cmap='gray')
        ax.set_title(f"{title}\nShape: {img_array.shape}\nRange: [{img_array.min()}, {img_array.max()}]")
        ax.axis('off')
        
        # Check for issues
        if img_array.std() < 1:
            ax.text(0.5, 0.5, 'BLANK IMAGE!', ha='center', va='center', 
                   transform=ax.transAxes, color='red', fontsize=16, fontweight='bold')
    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {e}', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

# Show malignant
for idx, (_, row) in enumerate(malignant_samples.iterrows()):
    load_and_show(axes[0, idx], row, f"Malignant {idx+1}")

# Show benign
for idx, (_, row) in enumerate(benign_samples.iterrows()):
    load_and_show(axes[1, idx], row, f"Benign {idx+1}")

plt.tight_layout()
plt.show()

print("\n⚠️  Visual check:")
print("   - Are images visible (not blank)?")
print("   - Do they show breast tissue?")
print("   - Is there clear difference between malignant/benign?")
print("   - Are value ranges reasonable (0-255 for PNG)?")

## Step 4: Check Preprocessing and Normalization

In [ ]:
import torch
from torchvision import transforms

# Load a sample image and check preprocessing
sample_row = train_df.iloc[0]
if 'png_path' in sample_row:
    img_path = os.path.join(PREPROCESSED_DIR, sample_row['png_path'])
else:
    img_path = os.path.join(PREPROCESSED_DIR, sample_row['image_path'])

img_pil = Image.open(img_path).convert('L')
img_array = np.array(img_pil)

print("Original PNG:")
print(f"  Shape: {img_array.shape}")
print(f"  Dtype: {img_array.dtype}")
print(f"  Range: [{img_array.min()}, {img_array.max()}]")
print(f"  Mean:  {img_array.mean():.2f}")
print(f"  Std:   {img_array.std():.2f}")

# Apply same preprocessing as training
img_tensor = torch.from_numpy(img_array).float() / 255.0
img_tensor = img_tensor.unsqueeze(0)

print("\nAfter /255.0 normalization:")
print(f"  Shape: {img_tensor.shape}")
print(f"  Dtype: {img_tensor.dtype}")
print(f"  Range: [{img_tensor.min():.4f}, {img_tensor.max():.4f}]")
print(f"  Mean:  {img_tensor.mean():.4f}")
print(f"  Std:   {img_tensor.std():.4f}")

# Resize
transform = transforms.Resize((224, 224))
img_resized = transform(img_tensor)

print("\nAfter resize to 224x224:")
print(f"  Shape: {img_resized.shape}")
print(f"  Range: [{img_resized.min():.4f}, {img_resized.max():.4f}]")
print(f"  Mean:  {img_resized.mean():.4f}")
print(f"  Std:   {img_resized.std():.4f}")

# Convert to 3-channel
img_3ch = img_resized.repeat(3, 1, 1)

print("\nAfter converting to 3-channel:")
print(f"  Shape: {img_3ch.shape}")
print(f"  Range: [{img_3ch.min():.4f}, {img_3ch.max():.4f}]")
print(f"  Mean:  {img_3ch.mean():.4f}")
print(f"  Std:   {img_3ch.std():.4f}")

print("\n⚠️  Expected for ResNet:")
print("   - Input should be in [0, 1] range ✓ (we're dividing by 255)")
print("   - Shape should be (3, 224, 224)")
print("   - ImageNet normalization NOT applied (we're using raw [0,1])")
print("\n⚠️  NOTE: ResNet was pretrained with ImageNet normalization!")
print("   ImageNet mean: [0.485, 0.456, 0.406]")
print("   ImageNet std:  [0.229, 0.224, 0.225]")
print("   We should probably apply this normalization!")

## Step 5: Check Label Distribution in Detail

In [ ]:
# Prepare metadata
def prepare_metadata(df):
    df = df.copy()
    if 'image_id' not in df.columns:
        df['image_id'] = df['png_path'].apply(lambda x: Path(x).stem)
    if 'patient_id' not in df.columns and 'study_id' in df.columns:
        df['patient_id'] = df['study_id']
    if 'breast_id' not in df.columns:
        if 'laterality' in df.columns:
            df['breast_id'] = df['patient_id'] + '_' + df['laterality']
        else:
            df['breast_id'] = df['patient_id'] + '_Unknown'
    return df

train_df = prepare_metadata(train_df)
val_df = prepare_metadata(val_df)

print("="*70)
print("LABEL ANALYSIS")
print("="*70)

# Image-level
print("\nImage-Level:")
print(f"  Train: {(train_df['label'] == 1).sum()} malignant / {(train_df['label'] == 0).sum()} benign")
print(f"  Val:   {(val_df['label'] == 1).sum()} malignant / {(val_df['label'] == 0).sum()} benign")

# Breast-level
print("\nBreast-Level:")
for name, df in [('Train', train_df), ('Val', val_df)]:
    breast_labels = df.groupby('breast_id')['label'].max()
    print(f"  {name}: {(breast_labels == 1).sum()} malignant / {(breast_labels == 0).sum()} benign breasts")

# Patient-level
print("\nPatient-Level:")
for name, df in [('Train', train_df), ('Val', val_df)]:
    patient_labels = df.groupby('patient_id')['label'].max()
    print(f"  {name}: {(patient_labels == 1).sum()} malignant / {(patient_labels == 0).sum()} benign patients")

# Check for label leakage
print("\n" + "="*70)
print("DATA LEAKAGE CHECK")
print("="*70)
train_patients = set(train_df['patient_id'].unique())
val_patients = set(val_df['patient_id'].unique())
overlap = train_patients & val_patients

if len(overlap) > 0:
    print(f"\n❌ DATA LEAKAGE DETECTED!")
    print(f"   {len(overlap)} patients appear in BOTH train and val sets!")
    print(f"   This inflates validation metrics!")
else:
    print(f"\n✅ No patient overlap between train and val (good!)")

## Step 6: Calculate Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
labels = train_df['label'].values
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=labels)

print("="*70)
print("CLASS WEIGHT RECOMMENDATION")
print("="*70)
print(f"\nClass weights (balanced):")
print(f"  Benign (0):    {class_weights[0]:.4f}")
print(f"  Malignant (1): {class_weights[1]:.4f}")
print(f"\nWeight ratio: {class_weights[1]/class_weights[0]:.2f}x more weight on malignant")

print("\n💡 RECOMMENDATION:")
print("   Use weighted loss to handle class imbalance:")
print(f"   pos_weight = torch.tensor([{class_weights[1]/class_weights[0]:.4f}])")
print(f"   criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)")

## Step 7: Test Model Forward Pass

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class ResNet50Classifier(nn.Module):
    def __init__(self, unfreeze_fraction=1.0, dropout_rate=0.0):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(2048, 1)
    
    def forward(self, x):
        features = self.features(x)
        features = self.avgpool(features)
        features = torch.flatten(features, 1)
        features = self.dropout(features)
        logits = self.classifier(features)
        probs = torch.sigmoid(logits).squeeze(1)
        return probs

# Create model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ResNet50Classifier(unfreeze_fraction=0.5, dropout_rate=0.3).to(device)

# Test forward pass
test_batch = torch.randn(4, 3, 224, 224).to(device)
print("Testing model forward pass...")
print(f"Input shape: {test_batch.shape}")

model.eval()
with torch.no_grad():
    output = model(test_batch)

print(f"Output shape: {output.shape}")
print(f"Output values: {output}")
print(f"Output range: [{output.min():.4f}, {output.max():.4f}]")

if output.min() < 0 or output.max() > 1:
    print("\n❌ ERROR: Output not in [0, 1] range!")
else:
    print("\n✅ Model forward pass works correctly")

# Check if model is learning anything
print("\nInitial predictions (random):")
print(f"  Mean: {output.mean():.4f}")
print(f"  Std:  {output.std():.4f}")
print("\n💡 Untrained model should output ~0.5 (random)")

## Summary and Recommendations

In [ ]:
print("="*70)
print("DIAGNOSIS SUMMARY")
print("="*70)

issues = []
recommendations = []

# Check dataset size
if len(train_df) < 500:
    issues.append("🔴 CRITICAL: Very small training set")
    recommendations.append("Use more data or simpler model (freeze more layers)")

if (train_df['label'] == 1).sum() < 50:
    issues.append("🔴 CRITICAL: Very few malignant samples")
    recommendations.append("Collect more data or use heavy augmentation")

# Check class imbalance
imbalance_ratio = (train_df['label'] == 0).sum() / max(1, (train_df['label'] == 1).sum())
if imbalance_ratio > 5:
    issues.append(f"🟡 WARNING: Severe class imbalance ({imbalance_ratio:.1f}:1)")
    recommendations.append("Use weighted loss or focal loss")

# Check normalization
issues.append("🟡 WARNING: Not using ImageNet normalization")
recommendations.append("Apply ImageNet mean/std normalization to inputs")

print("\n📋 Issues Found:")
for issue in issues:
    print(f"   {issue}")

print("\n💡 Recommendations:")
for i, rec in enumerate(recommendations, 1):
    print(f"   {i}. {rec}")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("\n1. Fix ImageNet normalization")
print("2. Use class weights for imbalanced data")
print("3. If dataset is small (<500 images):")
print("   - Freeze more layers (unfreeze_fraction=0.2)")
print("   - Reduce dropout (0.1-0.2)")
print("   - Increase augmentation (strength=0.8-1.0)")
print("4. Monitor training loss - should decrease!")
print("5. If still poor, check if labels are correct")
print("="*70)